In [0]:
from pyspark.sql.functions import col as spark_col, sum as spark_sum
from pyspark.sql import Row
from datetime import datetime
from time import perf_counter
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [0]:
{
    "check_name": "",
    "check_category": "",
    "status": "",
    "source_table": "",
    "target_table": "",
    "key_columns": "",
    "compared_columns": "",
    "src_count": None,
    "tgt_count": None,
    "mismatch_count": None,
    "src_only_count": None,
    "tgt_only_count": None,
    "value_mismatch_count": None,
    "duplicate_count": None,
    "null_count": None,
    "execution_time_ms": None,
    "remarks": ""
}

In [0]:
def row_count_check(spark, src_table, tgt_table):
    start = perf_counter()
    src_count = spark.table(src_table).count()
    tgt_count = spark.table(tgt_table).count()
    elapsed_ms = int((perf_counter() - start) * 1000)

    return {
        "check_name": "row_count_check",
        "check_category": "volume",
        "status": "PASS" if src_count == tgt_count else "FAIL",
        "source_table": src_table,
        "target_table": tgt_table,
        "key_columns": "",
        "compared_columns": "",
        "src_count": src_count,
        "tgt_count": tgt_count,
        "mismatch_count": abs(src_count - tgt_count),
        "src_only_count": None,
        "tgt_only_count": None,
        "value_mismatch_count": None,
        "duplicate_count": None,
        "null_count": None,
        "execution_time_ms": elapsed_ms,
        "remarks": ""
    }

In [0]:
def duplicate_check(spark, table_name, key_cols):
    start = perf_counter()
    df = spark.table(table_name)

    if isinstance(key_cols, str):
        key_cols = [key_cols]

    dup_count = df.groupBy(key_cols).count().filter("count > 1").count()
    elapsed_ms = int((perf_counter() - start) * 1000)

    return {
        "check_name": "duplicate_check",
        "check_category": "uniqueness",
        "status": "PASS" if dup_count == 0 else "FAIL",
        "source_table": table_name,
        "target_table": "",
        "key_columns": ",".join(key_cols),
        "compared_columns": "",
        "src_count": df.count(),
        "tgt_count": None,
        "mismatch_count": None,
        "src_only_count": None,
        "tgt_only_count": None,
        "value_mismatch_count": None,
        "duplicate_count": dup_count,
        "null_count": None,
        "execution_time_ms": elapsed_ms,
        "remarks": f"Checked on {key_cols}"
    }

In [0]:
def groupby_check(spark, src_table, tgt_table, group_cols):
    start = perf_counter()
    src_df = spark.table(src_table)
    tgt_df = spark.table(tgt_table)

    src_group = src_df.groupBy(group_cols).count()
    tgt_group = tgt_df.groupBy(group_cols).count()

    mismatch = src_group.subtract(tgt_group).count() + tgt_group.subtract(src_group).count()
    elapsed_ms = int((perf_counter() - start) * 1000)

    return {
        "check_name": "groupby_check",
        "check_category": "distribution",
        "status": "PASS" if mismatch == 0 else "FAIL",
        "source_table": src_table,
        "target_table": tgt_table,
        "key_columns": ",".join(group_cols),
        "compared_columns": "count",
        "src_count": src_df.count(),
        "tgt_count": tgt_df.count(),
        "mismatch_count": mismatch,
        "src_only_count": None,
        "tgt_only_count": None,
        "value_mismatch_count": mismatch,
        "duplicate_count": None,
        "null_count": None,
        "execution_time_ms": elapsed_ms,
        "remarks": f"grouped by {group_cols}"
    }

In [0]:
def null_check(spark, table_name):
    start = perf_counter()
    df = spark.table(table_name)
    cols = df.columns

    # Dynamic SQL
    null_expr = ", ".join([
        f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
        for c in cols
    ])

    query = f"SELECT {null_expr} FROM {table_name}"

    result = spark.sql(query).collect()[0].asDict()

    # Pure Python sum (IMPORTANT)
    total_nulls = sum(v for v in result.values() if v is not None)
    elapsed_ms = int((perf_counter() - start) * 1000)

    return {
        "check_name": "null_check",
        "check_category": "completeness",
        "status": "PASS" if total_nulls == 0 else "FAIL",
        "source_table": table_name,
        "target_table": "",
        "key_columns": "",
        "compared_columns": ",".join(cols),
        "src_count": df.count(),
        "tgt_count": None,
        "mismatch_count": None,
        "src_only_count": None,
        "tgt_only_count": None,
        "value_mismatch_count": None,
        "duplicate_count": None,
        "null_count": total_nulls,
        "execution_time_ms": elapsed_ms,
        "remarks": "SQL-based null check"
    }

In [0]:
def referential_check(spark, child_table, parent_table, join_col):
    start = perf_counter()
    child_df = spark.table(child_table)
    parent_df = spark.table(parent_table)

    invalid_df = child_df.join(parent_df, join_col, "left_anti")
    invalid_count = invalid_df.count()
    elapsed_ms = int((perf_counter() - start) * 1000)

    return {
        "check_name": "referential_check",
        "check_category": "integrity",
        "status": "PASS" if invalid_count == 0 else "FAIL",
        "source_table": child_table,
        "target_table": parent_table,
        "key_columns": join_col,
        "compared_columns": join_col,
        "src_count": child_df.count(),
        "tgt_count": parent_df.count(),
        "mismatch_count": invalid_count,
        "src_only_count": invalid_count,
        "tgt_only_count": None,
        "value_mismatch_count": None,
        "duplicate_count": None,
        "null_count": None,
        "execution_time_ms": elapsed_ms,
        "remarks": f"{join_col} integrity check"
    }

In [ ]:
def column_level_comparison_check(spark, src_table, tgt_table, key_cols, compare_cols=None):
    start = perf_counter()
    src_df = spark.table(src_table)
    tgt_df = spark.table(tgt_table)

    if isinstance(key_cols, str):
        key_cols = [key_cols]

    common_cols = [c for c in src_df.columns if c in tgt_df.columns]

    if compare_cols is None:
        compare_cols = [c for c in common_cols if c not in key_cols]

    src_selected = src_df.select(*(key_cols + compare_cols))
    tgt_selected = tgt_df.select(*(key_cols + compare_cols))

    src_alias = src_selected.alias("src")
    tgt_alias = tgt_selected.alias("tgt")

    join_condition = [spark_col(f"src.{k}") == spark_col(f"tgt.{k}") for k in key_cols]

    mismatch_condition = None
    for c in compare_cols:
        col_mismatch = (
            spark_col(f"src.{c}").isNull() != spark_col(f"tgt.{c}").isNull()
        ) | (
            spark_col(f"src.{c}").isNotNull()
            & spark_col(f"tgt.{c}").isNotNull()
            & (spark_col(f"src.{c}") != spark_col(f"tgt.{c}"))
        )
        mismatch_condition = col_mismatch if mismatch_condition is None else (mismatch_condition | col_mismatch)

    compared_df = src_alias.join(tgt_alias, join_condition, "inner")
    value_mismatch_count = 0 if mismatch_condition is None else compared_df.filter(mismatch_condition).count()

    src_only_count = src_selected.join(tgt_selected, key_cols, "left_anti").count()
    tgt_only_count = tgt_selected.join(src_selected, key_cols, "left_anti").count()

    total_mismatch = value_mismatch_count + src_only_count + tgt_only_count
    elapsed_ms = int((perf_counter() - start) * 1000)

    return {
        "check_name": "column_level_comparison_check",
        "check_category": "content",
        "status": "PASS" if total_mismatch == 0 else "FAIL",
        "source_table": src_table,
        "target_table": tgt_table,
        "key_columns": ",".join(key_cols),
        "compared_columns": ",".join(compare_cols),
        "src_count": src_df.count(),
        "tgt_count": tgt_df.count(),
        "mismatch_count": total_mismatch,
        "src_only_count": src_only_count,
        "tgt_only_count": tgt_only_count,
        "value_mismatch_count": value_mismatch_count,
        "duplicate_count": None,
        "null_count": None,
        "execution_time_ms": elapsed_ms,
        "remarks": "Column-level source-target validation"
    }

In [0]:
def log_results(spark, results):
    from datetime import datetime as dt
    from pyspark.sql import Row
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType

    schema = StructType([
        StructField("check_name",           StringType(),   True),
        StructField("check_category",       StringType(),   True),
        StructField("status",               StringType(),   True),
        StructField("source_table",         StringType(),   True),
        StructField("target_table",         StringType(),   True),
        StructField("key_columns",          StringType(),   True),
        StructField("compared_columns",     StringType(),   True),
        StructField("src_count",            IntegerType(),  True),
        StructField("tgt_count",            IntegerType(),  True),
        StructField("mismatch_count",       IntegerType(),  True),
        StructField("src_only_count",       IntegerType(),  True),
        StructField("tgt_only_count",       IntegerType(),  True),
        StructField("value_mismatch_count", IntegerType(),  True),
        StructField("duplicate_count",      IntegerType(),  True),
        StructField("null_count",           IntegerType(),  True),
        StructField("execution_time_ms",    IntegerType(),  True),
        StructField("remarks",              StringType(),   True),
        StructField("run_id",               StringType(),   True),
        StructField("timestamp",            TimestampType(), True),
    ])

    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    run_id = f"{current_user}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    standardized_rows = []

    for r in results:
        standardized_rows.append(Row(
            check_name = r.get("check_name"),
            check_category = r.get("check_category") or "",
            status = r.get("status"),
            source_table = r.get("source_table") or "",
            target_table = r.get("target_table") or "",
            key_columns = r.get("key_columns") or "",
            compared_columns = r.get("compared_columns") or "",
            src_count = r.get("src_count") or 0,
            tgt_count = r.get("tgt_count") or 0,
            mismatch_count = r.get("mismatch_count") or 0,
            src_only_count = r.get("src_only_count") or 0,
            tgt_only_count = r.get("tgt_only_count") or 0,
            value_mismatch_count = r.get("value_mismatch_count") or 0,
            duplicate_count = r.get("duplicate_count") or 0,
            null_count = r.get("null_count") or 0,
            execution_time_ms = r.get("execution_time_ms") or 0,
            remarks = r.get("remarks") or "",
            run_id = run_id,
            timestamp = dt.now()
        ))

    df = spark.createDataFrame(standardized_rows, schema=schema)
    df.write.mode("append").saveAsTable("project_training_amh.default.qa_validation_results")